# Arizona School Districts — Staff Directory Pipeline

For each district in the CSV:

1. **Discover schools** within the district (nav mining, sitemap, canonical `/schools` paths, subdomain detection, name-based deduplication)
2. **For each school** (plus the district homepage itself) — discover its staff directory URL
3. **Classify** the directory page (STAFF_DIRECTORY vs LOGIN vs NEWS vs junk)
4. **Extract** the people using the strategy that fits the page (JSON-LD → microdata → tables → cards → repeating blocks → mailto)

Outputs go to Google Drive so nothing is lost if Colab disconnects. `--resume` picks up where you left off.


## 1. Install dependencies and mount Google Drive


In [ ]:
!pip install -q beautifulsoup4 lxml requests tqdm

from google.colab import drive
drive.mount('/content/drive')

import os
OUT_DIR = '/content/drive/MyDrive/staffdir_az'
os.makedirs(OUT_DIR, exist_ok=True)
print('Outputs will go to:', OUT_DIR)


## 2. Upload the three required files

The next cell opens a file picker. Select all three at once:
- **`staffdir.zip`** — the package
- **`Universal-Staff-directory-Scraper-for-US-Schools--main__1_.zip`** — your legacy repo
- **`AZ_Districts_and_Site_Links_-_Sheet1__1_.csv`** — the AZ input


In [ ]:
from google.colab import files
print('Select: staffdir.zip, your legacy scraper zip, and the AZ CSV.')
uploaded = files.upload()
print()
print('Uploaded:')
for name in uploaded: print(' ', name)


## 3. Unpack and set up the working directory

Unzip the legacy scraper, then drop `staffdir/` inside it — required so `staffdir.adapt` can import `classify_lead` and `infer_role`.


In [ ]:
import os, zipfile, shutil, glob

WORK = '/content/scraper'
if os.path.exists(WORK): shutil.rmtree(WORK)
os.makedirs(WORK)

# Find whichever zip has master_staff_directory_scraper.
legacy_zip = None
for f in glob.glob('/content/*.zip'):
    with zipfile.ZipFile(f) as zf:
        if any('master_staff_directory_scraper' in n for n in zf.namelist()):
            legacy_zip = f; break
if legacy_zip is None:
    raise SystemExit('No legacy scraper zip found. Upload it in cell 2.')
print('Legacy repo zip:', legacy_zip)

with zipfile.ZipFile(legacy_zip) as zf:
    zf.extractall(WORK)

# Flatten if the zip contained a single top-level folder.
inner = [d for d in os.listdir(WORK) if os.path.isdir(os.path.join(WORK, d))]
if len(inner) == 1:
    for name in os.listdir(os.path.join(WORK, inner[0])):
        shutil.move(os.path.join(WORK, inner[0], name),
                    os.path.join(WORK, name))
    shutil.rmtree(os.path.join(WORK, inner[0]))

# Unzip staffdir.zip INSIDE WORK.
if not os.path.exists('/content/staffdir.zip'):
    raise SystemExit('staffdir.zip not uploaded. Upload it in cell 2.')
with zipfile.ZipFile('/content/staffdir.zip') as zf:
    zf.extractall(WORK)

print()
print('Working directory contents:')
for f in sorted(os.listdir(WORK))[:20]: print(' ', f)
assert os.path.exists(os.path.join(WORK, 'master_staff_directory_scraper.py'))
assert os.path.exists(os.path.join(WORK, 'staffdir', '__init__.py'))
print()
print('OK - staffdir/ sits next to master_staff_directory_scraper.py')


## 4. Quick sanity check — 33 tests should pass


In [ ]:
%cd /content/scraper
!python -m staffdir.tests_smoke
!python -m staffdir.tests_pipeline


## 5. Load and normalize the Arizona CSV

The uploaded CSV has columns `School Districts, Website` and an unnamed status column. We rename them to `name, website`, drop rows without a URL, and skip districts already marked `Done` in the source sheet.


In [ ]:
import glob, pandas as pd, os, sys

csvs = [c for c in glob.glob('/content/*.csv')
         if 'az' in c.lower() or 'arizona' in c.lower() or 'district' in c.lower()]
if not csvs:
    print('No AZ CSV in /content. Files present:')
    for f in glob.glob('/content/*'): print(' ', f)
    sys.exit()
src_csv = csvs[0]
print('Source CSV:', src_csv)

df = pd.read_csv(src_csv)
print('Original columns:', list(df.columns))
print('Row count:', len(df))
df.head(5)


In [ ]:
df = df.rename(columns={'School Districts': 'name', 'Website': 'website'})

df = df[df['website'].notna()].copy()
df['website'] = df['website'].str.strip().str.rstrip('/')
df = df[df['website'].str.startswith(('http://', 'https://'))]

# Skip rows already marked 'Done' in the source sheet.
if df.shape[1] >= 3:
    status_col = [c for c in df.columns if c not in ('name', 'website')][0]
    n_done = (df[status_col].fillna('').str.lower() == 'done').sum()
    if n_done:
        print(f'Skipping {n_done} rows marked "Done" in the source sheet.')
        df = df[df[status_col].fillna('').str.lower() != 'done']

df = df[['name', 'website']].reset_index(drop=True)
print(f'Districts to process: {len(df)}')
df.to_csv('/content/scraper/az_input.csv', index=False)
df.head(10)


## 6. Pilot run — 5 districts first

Before running on all 175, run 5 to sanity-check speed and coverage. Tuning knobs on this run:

- `--workers 3` — 3 districts in parallel
- `--inner-workers 8` — 8 schools per district in parallel (this is the speed fix that keeps a district with 20 schools from taking 12 minutes)
- `--time-budget 120` — hard cap: if a district takes >2 min, cancel remaining schools and move on
- `--max-schools-per-district 25` — most AZ districts have <20 real schools; this cap prevents runaway discovery

If this pilot shows a district with wildly high school counts, the school-discovery filter needs tuning for that specific site — inspect the `warnings` column.


In [ ]:
%cd /content/scraper
!python -m staffdir az_input.csv \
  --district \
  --out-prefix az_pilot \
  --workers 3 \
  --inner-workers 8 \
  --time-budget 120 \
  --max-schools-per-district 25 \
  --limit 5


In [ ]:
import pandas as pd
schools = pd.read_csv('/content/scraper/az_pilot_schools.csv')
leads   = pd.read_csv('/content/scraper/az_pilot_leads.csv')

print(f'Schools rows: {len(schools)}   Leads rows: {len(leads)}')
print()
print('Per-district summary:')
print(schools.groupby('district_name').agg(
    schools_scanned=('school_name', 'nunique'),
    dirs_extracted=('directory_url', lambda s: (s.fillna('') != '').sum()),
    total_persons=('n_persons_extracted', 'sum'),
))
print()
print('Sample leads:')
leads[['district','school','first_name','last_name','email','title','lead_type']].head(15)


### Diagnostic — schools with no data

Any row with blank `directory_url` or `n_persons_extracted == 0`. Look at `warnings` to see why.


In [ ]:
no_data = schools[(schools['directory_url'].fillna('') == '') |
                   (schools['n_persons_extracted'] == 0)]
print(f'{len(no_data)} school rows produced no data')
no_data[['district_name','school_name','school_url','warnings']].head(20)


## 7. Full run — all districts

When the pilot looks right, run the full ~175. Outputs go to Drive so a disconnect doesn't lose progress. `--resume` skips districts already in the output CSV.

Estimated time at these settings: ~30-90 minutes depending on how many schools each district has and how responsive their sites are.


In [ ]:
%cd /content/scraper
OUT_PREFIX = f'{OUT_DIR}/az_full'
!python -m staffdir az_input.csv \
  --district \
  --out-prefix "$OUT_PREFIX" \
  --workers 4 \
  --inner-workers 8 \
  --time-budget 180 \
  --max-schools-per-district 30 \
  --resume


## 8. Summarize results


In [ ]:
import pandas as pd
schools = pd.read_csv(f'{OUT_DIR}/az_full_schools.csv')
leads   = pd.read_csv(f'{OUT_DIR}/az_full_leads.csv')

print(f'Total districts processed: {schools["district_name"].nunique()}')
print(f'Total schools scanned:     {schools["school_name"].nunique()}')
print(f'Total leads extracted:     {len(leads)}')
print(f'Leads with valid email:    {leads["email"].fillna("").str.contains("@").sum()}')
print()
print('Leads by lead_type:')
print(leads['lead_type'].value_counts())
print()
print('Top 10 districts by lead count:')
print(leads.groupby('district').size().sort_values(ascending=False).head(10))
print()
print('Districts with ZERO leads (may need JS render or manual inspection):')
zero = set(schools['district_name'].unique()) - set(leads['district'].unique())
for d in sorted(zero)[:20]: print(' ', d)


## 9. Deduplicate on email (optional final cleanup)


In [ ]:
import pandas as pd
leads = pd.read_csv(f'{OUT_DIR}/az_full_leads.csv')

leads['email'] = leads['email'].fillna('').str.lower().str.strip()
with_email = leads[leads['email'].str.contains('@')]

with_email = with_email.sort_values('overall_confidence', ascending=False)
deduped = with_email.drop_duplicates(subset=['email'], keep='first')

out = f'{OUT_DIR}/az_full_leads_deduped.csv'
deduped.to_csv(out, index=False)
print(f'Before dedup: {len(with_email)}   After dedup: {len(deduped)}')
print(f'Saved to: {out}')
deduped.head()
